# Day 12 — Gold Dimensional: Fact Tables

Builds all 7 buildable fact tables from Silver sources, joining to Gold dim surrogate keys.
Facts not buildable from source are omitted (FactFleetUtilisation, FactInvoice amounts).

| Fact Table | Source | Grain | Notes |
|---|---|---|---|
| FactChargingSession | api/sessions + payments + dim keys | 1 row/session | reconciliation_status = MATCH/MISMATCH |
| FactEnergyConsumption | api/sessions aggregated | 1 row/charger/hour | energy_kwh, duration_min |
| FactPayments | api/payments | 1 row/payment | gst=amount*0.10, refund_flag from status |
| FactMaintenance | api/maintenance_events + realtime/charging_sessions | 1 row/event | API + realtime blob maintenance combined |
| FactDeviceTelemetry | realtime/charging_sessions | 1 row/session | peak_kw, raw_device_temp_c, signal_strength_dbm |
| FactComplaints | api/complaints | 1 row/complaint | resolution_days from updated_at - created_at |
| FactStationUtilisation | api/sessions aggregated | 1 row/station/hour | session_count, total_energy_kwh, utilisation_pct |

In [ ]:
# Cell 1 — Imports
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime, timezone

print('Imports OK')

In [ ]:
# Cell 2 — Constants
SILVER_API = '/Volumes/dbw_ev_intelligence_dev/default/silver-volume/api'
SILVER_RT  = '/Volumes/dbw_ev_intelligence_dev/default/silver-volume/realtime'
GOLD_DIM   = '/Volumes/dbw_ev_intelligence_dev/default/gold-volume/dimensional/dims'
GOLD_FACT  = '/Volumes/dbw_ev_intelligence_dev/default/gold-volume/dimensional/facts'
PIPELINE   = 'pl_gold_dimensional_facts_v1'
RUN_TS     = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')

def read_silver(name):
    return spark.read.format('delta').load(f'{SILVER_API}/{name}')

def read_dim(name):
    return spark.read.format('delta').load(f'{GOLD_DIM}/{name}')

def write_fact(df, fact_name, partition_cols=None):
    path = f'{GOLD_FACT}/{fact_name}'
    writer = df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true')
    if partition_cols:
        writer = writer.partitionBy(*partition_cols)
    writer.save(path)
    print(f'  {fact_name:<30} {df.count():>8} rows  -> {path}')

print(f'Gold facts : {GOLD_FACT}')
print(f'RUN_TS     : {RUN_TS}')

In [ ]:
# Cell 3 — Load dim surrogate key lookups (current rows only)
# We resolve natural keys → surrogate keys for each fact join.

dim_station  = read_dim('DimStation').select('station_key',  'station_id')
dim_charger  = read_dim('DimCharger').filter(F.col('is_current') == True).select('charger_key',  'charger_id')
dim_customer = read_dim('DimCustomer').filter(F.col('is_current') == True).select('customer_key', 'customer_id')
dim_vehicle  = read_dim('DimVehicle').select('vehicle_key',  'vehicle_id')
dim_tariff   = read_dim('DimTariff').filter(F.col('is_current') == True).select('tariff_key',   'tariff_id')
dim_employee = read_dim('DimEmployee').select('employee_key', 'employee_id')
dim_card     = read_dim('DimChargeCard').select('card_key', 'card_id')

# DimTime key helper: given a timestamp column, derive time_key as YYYYMMDDHHH string matching DimTime.time_key
def time_key_expr(ts_col):
    return F.concat_ws('',
        F.year(ts_col).cast('string'),
        F.lpad(F.month(ts_col).cast('string'),      2, '0'),
        F.lpad(F.dayofmonth(ts_col).cast('string'),  2, '0'),
        F.lpad(F.hour(ts_col).cast('string'),        2, '0'),
    )

print('Dim lookup tables loaded')

In [ ]:
# Cell 4 — FactChargingSession
# Grain: 1 row per charging session.
# Joins API sessions to payments to compare expected tariff cost vs actual payment amount.
# reconciliation_status: MATCH if payment ≈ tariff cost (within 1 cent), MISMATCH otherwise, UNMATCHED if no payment.

sessions_raw = read_silver('sessions')
payments_raw = read_silver('payments')

# Aggregate payment amount per session (sum in case of split payments)
payments_agg = (
    payments_raw
    .groupBy('session_id')
    .agg(
        F.sum('amount').alias('total_payment_aud'),
        F.count('payment_id').alias('payment_count'),
        F.collect_set('status').alias('payment_statuses'),
    )
)

fact_session = (
    sessions_raw
    # Surrogate key lookups
    .join(dim_station,  on='station_id',  how='left')
    .join(dim_charger,  on='charger_id',  how='left')
    .join(dim_customer, on='customer_id', how='left')
    .join(dim_vehicle,  on='vehicle_id',  how='left')
    .join(dim_tariff,   on='tariff_id',   how='left')
    # Payment data
    .join(payments_agg, on='session_id',  how='left')
    # Derived metrics
    .withColumn('start_time_key', time_key_expr(F.col('start_time')))
    .withColumn('session_date',   F.to_date('start_time'))
    .withColumn('session_year',   F.year('start_time'))
    .withColumn('session_month',  F.month('start_time'))
    # Tariff-based expected cost
    .withColumn('expected_cost_aud',
        F.when(
            F.col('price_per_kwh').isNotNull() & F.col('energy_kwh').isNotNull(),
            (F.col('price_per_kwh') * F.col('energy_kwh')
             + F.coalesce(F.col('price_per_min'), F.lit(0)) * F.col('duration_min')
            ).cast('decimal(10,2)')
        )
    )
    # Reconciliation: compare expected vs actual payment
    .withColumn('reconciliation_status',
        F.when(F.col('total_payment_aud').isNull(), F.lit('UNMATCHED'))
        .when(
            F.col('expected_cost_aud').isNotNull() &
            (F.abs(F.col('total_payment_aud') - F.col('expected_cost_aud')) < F.lit(0.01)),
            F.lit('MATCH')
        )
        .otherwise(F.lit('MISMATCH'))
    )
    .withColumn('cost_per_kwh',
        F.when(
            F.col('energy_kwh').isNotNull() & (F.col('energy_kwh') > 0) &
            F.col('total_payment_aud').isNotNull(),
            (F.col('total_payment_aud') / F.col('energy_kwh')).cast('decimal(8,4)')
        )
    )
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select(
        'session_id',
        'station_key', 'charger_key', 'customer_key', 'vehicle_key', 'tariff_key',
        'start_time_key',
        'session_date', 'session_year', 'session_month',
        F.col('start_time').alias('session_start_ts'),
        F.col('end_time').alias('session_end_ts'),
        F.col('duration_min'),
        F.col('energy_kwh'),
        F.col('price_per_kwh').alias('tariff_rate_per_kwh'),
        'expected_cost_aud',
        'total_payment_aud',
        'payment_count',
        'cost_per_kwh',
        'reconciliation_status',
        F.col('status').alias('session_status'),
        'gold_ingested_at', 'gold_pipeline',
    )
)

write_fact(fact_session, 'FactChargingSession', partition_cols=['session_year', 'session_month'])
print('\nReconciliation status distribution:')
fact_session.groupBy('reconciliation_status').count().show()

In [ ]:
# Cell 5 — FactEnergyConsumption
# Grain: 1 row per charger per hour.
# Aggregates sessions by the start hour of each session.

fact_energy = (
    sessions_raw
    .join(dim_charger,  on='charger_id',  how='left')
    .join(dim_station,  on='station_id',  how='left')
    .withColumn('session_hour_ts',
        F.date_trunc('hour', F.col('start_time'))
    )
    .withColumn('time_key', time_key_expr(F.col('start_time')))
    .groupBy('charger_key', 'charger_id', 'station_key', 'station_id',
             'session_hour_ts', 'time_key')
    .agg(
        F.count('session_id').alias('session_count'),
        F.sum('energy_kwh').cast('decimal(12,4)').alias('total_energy_kwh'),
        F.avg('energy_kwh').cast('decimal(10,4)').alias('avg_energy_kwh'),
        F.sum('duration_min').cast('decimal(12,2)').alias('total_duration_min'),
        F.avg('duration_min').cast('decimal(8,2)').alias('avg_duration_min'),
    )
    .withColumn('consumption_year',  F.year('session_hour_ts'))
    .withColumn('consumption_month', F.month('session_hour_ts'))
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select(
        'charger_key', 'charger_id', 'station_key', 'station_id',
        'session_hour_ts', 'time_key',
        'consumption_year', 'consumption_month',
        'session_count', 'total_energy_kwh', 'avg_energy_kwh',
        'total_duration_min', 'avg_duration_min',
        'gold_ingested_at', 'gold_pipeline',
    )
)

write_fact(fact_energy, 'FactEnergyConsumption', partition_cols=['consumption_year', 'consumption_month'])

In [ ]:
# Cell 6 — FactPayments
# Grain: 1 row per payment transaction.
# gst_aud = amount * 0.10 (10% GST assumption for Australia).
# refund_flag = True if status is 'refunded' or 'reversed'.

fact_payments = (
    payments_raw
    .join(dim_customer,
          on=payments_raw['customer_id'] == dim_customer['customer_id'],
          how='left'
    ).drop(dim_customer['customer_id'])
    .join(dim_card,
          on=payments_raw['card_id'] == dim_card['card_id'],
          how='left'
    ).drop(dim_card['card_id'])
    .withColumn('payment_time_key', time_key_expr(F.col('payment_time')))
    .withColumn('payment_year',  F.year('payment_time'))
    .withColumn('payment_month', F.month('payment_time'))
    .withColumn('gst_aud',
        (F.col('amount') * F.lit(0.10)).cast('decimal(10,2)')
    )
    .withColumn('amount_ex_gst',
        (F.col('amount') * F.lit(0.9091)).cast('decimal(10,2)')   # amount / 1.10
    )
    .withColumn('refund_flag',
        F.when(
            F.lower(F.col('status')).isin('refunded', 'reversed', 'refund'),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select(
        'payment_id', 'session_id',
        'customer_key', 'customer_id',
        'card_key', 'card_id',
        'payment_time_key',
        'payment_year', 'payment_month',
        F.col('payment_time').alias('payment_ts'),
        F.col('amount').alias('amount_aud'),
        'amount_ex_gst', 'gst_aud',
        F.trim(F.col('method')).alias('payment_method'),
        F.trim(F.col('status')).alias('payment_status'),
        'refund_flag',
        'gold_ingested_at', 'gold_pipeline',
    )
)

write_fact(fact_payments, 'FactPayments', partition_cols=['payment_year', 'payment_month'])
print('\nPayment method breakdown:')
fact_payments.groupBy('payment_method', 'payment_status', 'refund_flag').count().orderBy('payment_method').show()

In [ ]:
# Cell 7 — FactMaintenance
# Grain: 1 row per maintenance event.
# Source: api/maintenance_events (scheduled/completed maintenance).
# resolution_hours: hours between created_at and resolved_at (NULL if not resolved).

maintenance_raw = read_silver('maintenance_events')

fact_maintenance = (
    maintenance_raw
    .join(dim_charger,  on='charger_id',  how='left')
    .join(dim_station,  on='station_id',  how='left')
    .join(dim_employee,
          on=maintenance_raw['assigned_employee_id'] == dim_employee['employee_id'],
          how='left'
    ).drop(dim_employee['employee_id'])
    .withColumn('event_time_key', time_key_expr(F.col('created_at')))
    .withColumn('event_year',  F.year('created_at'))
    .withColumn('event_month', F.month('created_at'))
    .withColumn('resolution_hours',
        F.when(
            F.col('resolved_at').isNotNull() & F.col('created_at').isNotNull(),
            ((F.col('resolved_at').cast('long') - F.col('created_at').cast('long')) / 3600
            ).cast('decimal(10,2)')
        )
    )
    .withColumn('is_resolved',
        F.when(F.col('resolved_at').isNotNull(), F.lit(True)).otherwise(F.lit(False))
    )
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select(
        'event_id',
        'charger_key', 'charger_id',
        'station_key', 'station_id',
        F.col('employee_key').alias('assigned_employee_key'),
        F.col('assigned_employee_id'),
        'event_time_key', 'event_year', 'event_month',
        F.col('created_at').alias('event_created_ts'),
        F.col('resolved_at').alias('event_resolved_ts'),
        'resolution_hours', 'is_resolved',
        F.trim(F.col('event_type')).alias('event_type'),
        F.trim(F.col('severity')).alias('severity'),
        F.trim(F.col('status')).alias('event_status'),
        'gold_ingested_at', 'gold_pipeline',
    )
)

write_fact(fact_maintenance, 'FactMaintenance', partition_cols=['event_year', 'event_month'])
print('\nMaintenance by severity:')
fact_maintenance.groupBy('severity', 'is_resolved').count().orderBy('severity').show()

In [ ]:
# Cell 8 — FactDeviceTelemetry
# Grain: 1 row per realtime charging session (IoT telemetry grain).
# Source: realtime/charging_sessions (blob Silver).
# Fields available in source: peak_kw, raw_device_temp_c, signal_strength_dbm, firmware_ver.
# voltage, current omitted — not in source.

rt_sessions = spark.read.format('delta').load(f'{SILVER_RT}/charging_sessions')

fact_telemetry = (
    rt_sessions
    .join(dim_charger,  on='charger_id',  how='left')
    .join(dim_station,  on='station_id',  how='left')
    .join(dim_customer, on='customer_id', how='left')
    .withColumn('telemetry_time_key', time_key_expr(F.col('plug_in_ts')))
    .withColumn('telemetry_year',  F.year('plug_in_ts'))
    .withColumn('telemetry_month', F.month('plug_in_ts'))
    # power_efficiency_pct: actual_kwh / (rated_kw × hours) × 100
    .withColumn('power_efficiency_pct',
        F.when(
            F.col('peak_kw').isNotNull() & (F.col('peak_kw') > 0)
            & F.col('duration_min').isNotNull() & (F.col('duration_min') > 0),
            (F.col('energy_kwh') / (F.col('peak_kw') * F.col('duration_min') / 60) * 100
            ).cast('decimal(6,2)')
        )
    )
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select(
        'session_id',
        'charger_key', 'charger_id',
        'station_key', 'station_id',
        'customer_key', 'customer_id',
        'telemetry_time_key', 'telemetry_year', 'telemetry_month',
        F.col('plug_in_ts').alias('session_start_ts'),
        F.col('plug_out_ts').alias('session_end_ts'),
        F.col('duration_min'),
        F.col('energy_kwh'),
        F.col('peak_kw').alias('peak_power_kw'),
        F.col('raw_device_temp_c').alias('device_temp_c'),
        F.col('signal_strength_dbm'),
        F.col('firmware_ver'),
        'power_efficiency_pct',
        F.trim(F.col('session_status')).alias('session_status'),
        'gold_ingested_at', 'gold_pipeline',
    )
)

write_fact(fact_telemetry, 'FactDeviceTelemetry', partition_cols=['telemetry_year', 'telemetry_month'])
print('\nAvg power efficiency by firmware:')
fact_telemetry.groupBy('firmware_ver').agg(
    F.count('session_id').alias('sessions'),
    F.round(F.avg('power_efficiency_pct'), 2).alias('avg_efficiency_pct'),
    F.round(F.avg('device_temp_c'), 2).alias('avg_temp_c'),
).orderBy('firmware_ver').show()

In [ ]:
# Cell 9 — FactComplaints
# Grain: 1 row per complaint.
# resolution_days: days from created_at to updated_at (proxy for resolution, since resolved_at is absent).
# is_resolved: status = 'resolved' or 'closed'.

complaints_raw = read_silver('complaints')

fact_complaints = (
    complaints_raw
    .join(dim_customer,
          on=complaints_raw['customer_id'] == dim_customer['customer_id'],
          how='left'
    ).drop(dim_customer['customer_id'])
    .join(dim_station,
          on=complaints_raw['station_id'] == dim_station['station_id'],
          how='left'
    ).drop(dim_station['station_id'])
    .withColumn('complaint_time_key', time_key_expr(F.col('created_at')))
    .withColumn('complaint_year',  F.year('created_at'))
    .withColumn('complaint_month', F.month('created_at'))
    .withColumn('resolution_days',
        F.when(
            F.col('updated_at').isNotNull() & F.col('created_at').isNotNull()
            & (F.lower(F.col('status')).isin('resolved', 'closed')),
            ((F.col('updated_at').cast('long') - F.col('created_at').cast('long')) / 86400
            ).cast('decimal(8,2)')
        )
    )
    .withColumn('is_resolved',
        F.when(
            F.lower(F.col('status')).isin('resolved', 'closed'),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select(
        'complaint_id',
        'customer_key', 'customer_id',
        'station_key', 'station_id',
        'complaint_time_key', 'complaint_year', 'complaint_month',
        F.col('created_at').alias('complaint_created_ts'),
        F.col('updated_at').alias('complaint_updated_ts'),
        F.trim(F.col('category')).alias('complaint_category'),
        F.trim(F.col('priority')).alias('priority'),
        F.trim(F.col('status')).alias('complaint_status'),
        'is_resolved', 'resolution_days',
        'gold_ingested_at', 'gold_pipeline',
    )
)

write_fact(fact_complaints, 'FactComplaints', partition_cols=['complaint_year', 'complaint_month'])
print('\nComplaints by category and resolution:')
fact_complaints.groupBy('complaint_category', 'is_resolved').agg(
    F.count('complaint_id').alias('count'),
    F.round(F.avg('resolution_days'), 1).alias('avg_resolution_days'),
).orderBy('complaint_category').show()

In [ ]:
# Cell 10 — FactStationUtilisation
# Grain: 1 row per station per hour (aggregated from API sessions).
# utilisation_pct: active session minutes in the hour / (total_chargers × 60) × 100, capped at 100.
# total_chargers joined from DimStation.

dim_station_full = read_dim('DimStation').select('station_key', 'station_id', 'total_chargers')

fact_utilisation = (
    sessions_raw
    .join(dim_station_full, on='station_id', how='left')
    .withColumn('session_hour_ts', F.date_trunc('hour', F.col('start_time')))
    .withColumn('time_key', time_key_expr(F.col('start_time')))
    .groupBy('station_key', 'station_id', 'session_hour_ts', 'time_key',
             'total_chargers')
    .agg(
        F.count('session_id').alias('session_count'),
        F.sum('energy_kwh').cast('decimal(12,4)').alias('total_energy_kwh'),
        F.sum('duration_min').cast('decimal(12,2)').alias('total_active_min'),
        F.countDistinct('charger_id').alias('active_charger_count'),
    )
    .withColumn('utilisation_year',  F.year('session_hour_ts'))
    .withColumn('utilisation_month', F.month('session_hour_ts'))
    # utilisation_pct = active_minutes / (chargers × 60 min/hour) × 100, capped at 100
    .withColumn('utilisation_pct',
        F.when(
            F.col('total_chargers').isNotNull() & (F.col('total_chargers') > 0),
            F.least(
                (F.col('total_active_min') / (F.col('total_chargers') * 60) * 100
                ).cast('decimal(6,2)'),
                F.lit(100.0).cast('decimal(6,2)')
            )
        )
    )
    .withColumn('gold_ingested_at', F.lit(RUN_TS).cast('timestamp'))
    .withColumn('gold_pipeline',    F.lit(PIPELINE))
    .select(
        'station_key', 'station_id',
        'session_hour_ts', 'time_key',
        'utilisation_year', 'utilisation_month',
        'total_chargers',
        'session_count',
        'active_charger_count',
        'total_energy_kwh',
        'total_active_min',
        'utilisation_pct',
        'gold_ingested_at', 'gold_pipeline',
    )
)

write_fact(fact_utilisation, 'FactStationUtilisation', partition_cols=['utilisation_year', 'utilisation_month'])
print('\nTop 5 stations by avg utilisation:')
fact_utilisation.groupBy('station_id').agg(
    F.round(F.avg('utilisation_pct'), 2).alias('avg_utilisation_pct'),
    F.count('session_hour_ts').alias('hours_with_sessions'),
).orderBy(F.col('avg_utilisation_pct').desc()).show(5)

In [ ]:
# Cell 11 — Run Summary
facts = [
    'FactChargingSession',
    'FactEnergyConsumption',
    'FactPayments',
    'FactMaintenance',
    'FactDeviceTelemetry',
    'FactComplaints',
    'FactStationUtilisation',
]
print('=' * 60)
print('GOLD DIMENSIONAL — FACT TABLES SUMMARY')
print('=' * 60)
for f in facts:
    path = f'{GOLD_FACT}/{f}'
    try:
        cnt = spark.read.format('delta').load(path).count()
        print(f'  {f:<30} {cnt:>8} rows')
    except Exception as e:
        print(f'  {f:<30} ERROR: {e}')
print(f'\nRUN_TS : {RUN_TS}')
print('\nSkipped (source data missing):')
print('  FactFleetUtilisation  — no telematics/fleet source data')
print('  FactInvoice amounts   — invoice PDFs not parsed into Silver')